In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd
import datetime

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from mlflow.models import infer_signature

from src.config import FEATURE_TABLE, MODEL_NAME, EXPERIMENT_NAME
from src.train import train_model
from src.registry import register_model, promote_model

exp = mlflow.set_experiment(EXPERIMENT_NAME)
print("Experiment ID:", exp.experiment_id)

df = spark.table(FEATURE_TABLE).toPandas()

y = df["Exited"]
X = pd.get_dummies(df.drop("Exited", axis=1))


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = train_model(X_train_scaled, y_train)

preds = model.predict(X_test_scaled)
acc = accuracy_score(y_test, preds)

print("Accuracy:", acc)

if acc < 0.80:
    raise Exception("Model rejected")

signature = infer_signature(X_test, pd.DataFrame(preds))

run_name = f"bank_churn_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(run_name=run_name) as run:
    run_id = run.info.run_id

    mlflow.log_metric("accuracy", acc)

    model_info = mlflow.sklearn.log_model(
        sk_model=model,
        name="bank_churn_model",
        signature=signature,
        input_example=X_test.head(5)
    )

print("MODEL LOGGED")

version = register_model(
    MODEL_NAME,
    model_info.model_uri,
    run_id
)

promote_model(MODEL_NAME, version)

print("MODEL Registered as PROD:", version)